# Condition Shift Baseline Notebook

이 노트북은 `condition_shift_baseline` 실험의 thin orchestrator다.

- Colab/서버에서 Git checkout 상태를 확인한다.
- 필요하면 prepared dataset bootstrap script를 호출한다.
- versioned Python runner를 실행한다.
- `summary.json`과 `log.txt`를 읽어 표와 간단 시각화를 보여준다.

실험 핵심 로직은 노트북에 두지 않고, notebook이 호출하는 `.py`는 repo에 versioned 상태로 유지한다.


## 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys
import threading
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents, Path('/content/ReGraM')]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and p.name in {'ReGraM'}), cwd)
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']
SEVERITY_ORDER = {'low': 0, 'medium': 1, 'high': 2}
DEFAULT_PATCHCORE_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def now_string():
    return datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S %Z')

def format_run_label(config):
    return f"{config['baseline']} / {config['category']}"

def display_title(title, body=None):
    text = f"## {title}"
    if body:
        text += f"\n\n{body}"
    display(Markdown(text))

def display_run_plan(run_configs):
    rows = []
    for idx, config in enumerate(run_configs, start=1):
        rows.append({
            'order': idx,
            'baseline': config['baseline'],
            'category': config['category'],
            'manifest_count': len(config['manifest_paths']),
            'summary_name': config['summary_path'].name,
            'device': config['device'],
            'wandb': 'on' if config.get('use_wandb') else 'off',
        })
    display(pd.DataFrame(rows))

def display_environment_summary():
    display_title('Environment Summary')
    display(pd.DataFrame([
        {'key': 'REPO_ROOT', 'value': str(REPO_ROOT)},
        {'key': 'EXP_ROOT', 'value': str(EXP_ROOT)},
        {'key': 'REPORT_ROOT', 'value': str(REPORT_ROOT)},
    ]))

def resolve_manifest_paths(manifest_names):
    manifest_paths = []
    for manifest_name in manifest_names:
        manifest_path = next(
            (root / manifest_name for root in MANIFEST_ROOT_CANDIDATES if (root / manifest_name).exists()),
            None,
        )
        if manifest_path is None:
            searched = [str(root / manifest_name) for root in MANIFEST_ROOT_CANDIDATES]
            raise FileNotFoundError(f'Manifest not found. searched={searched}')
        manifest_paths.append(manifest_path)
    return manifest_paths

def stream_subprocess(command, cwd, show_output=True):
    process = subprocess.Popen(
        command,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        bufsize=1,
    )
    stdout_lines = []
    stderr_lines = []

    def pump(pipe, prefix, sink):
        try:
            for line in iter(pipe.readline, ''):
                if not line:
                    break
                sink.append(line)
                if show_output:
                    print(f"{prefix} {line.rstrip()}", flush=True)
        finally:
            pipe.close()

    stdout_thread = threading.Thread(target=pump, args=(process.stdout, '[stdout]', stdout_lines))
    stderr_thread = threading.Thread(target=pump, args=(process.stderr, '[stderr]', stderr_lines))
    stdout_thread.start()
    stderr_thread.start()
    returncode = process.wait()
    stdout_thread.join()
    stderr_thread.join()
    return {
        'returncode': returncode,
        'stdout': ''.join(stdout_lines),
        'stderr': ''.join(stderr_lines),
    }

def print_output_tail(name, text, tail_lines):
    lines = [line for line in text.splitlines() if line.strip()]
    if not lines:
        print(f"[{name}] <empty>")
        return
    tail = lines[-tail_lines:]
    print(f"[{name} tail] showing last {len(tail)} line(s)")
    for line in tail:
        print(line)

run_history = []

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Git Checkout

Colab에서는 먼저 repo를 Git으로 clone 또는 pull 해서 notebook이 사용할 `.py`를 최신 상태로 맞춘다.


In [ ]:
REPO_URL = 'https://github.com/outSeop/ReGraM.git'
REPO_DIR = Path('/content/ReGraM')
git_cmd = (
    f'if [ -d "{REPO_DIR}/.git" ]; then git -C "{REPO_DIR}" pull --ff-only; '
    f'else git clone "{REPO_URL}" "{REPO_DIR}"; fi'
)
print(git_cmd)
subprocess.run(['bash', '-lc', git_cmd], check=True)

REPO_ROOT = REPO_DIR.resolve()
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']

print('updated REPO_ROOT =', REPO_ROOT)
print('updated EXP_ROOT =', EXP_ROOT)


## Dataset load

In [ ]:
from pathlib import Path
import subprocess

drive_tar = Path("/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz")
runtime_tar = Path("/content/mvtec_loco_anomaly_detection.tar.gz")
runtime_row = Path("/content/ReGraM/data/row")
runtime_root = runtime_row / "mvtec_loco_anomaly_detection"

print("drive_tar exists:", drive_tar.exists())
print("runtime_root exists:", runtime_root.exists())

runtime_row.mkdir(parents=True, exist_ok=True)

if not runtime_root.exists():
    if not runtime_tar.exists():
        subprocess.run(["cp", str(drive_tar), str(runtime_tar)], check=True)
    subprocess.run(
        ["tar", "-xf", str(runtime_tar), "-C", str(runtime_row)],
        check=True,
    )

print("done")
print(runtime_root.exists(), runtime_root)


## Optional Dataset Bootstrap

Git checkout 이후 prepared dataset이나 작은 reports 자산이 필요하면 아래처럼 별도 Python bootstrap script를 호출한다.
코드 동기화는 bootstrap이 아니라 Git이 담당한다.


In [ ]:
bootstrap_cmd = [
    sys.executable,
    str(EXP_ROOT / 'colab' / 'bootstrap_runtime.py'),
    '--dry-run',
]
print(' '.join(bootstrap_cmd))
subprocess.run(bootstrap_cmd, cwd=REPO_ROOT, check=True)


## Runner Orchestration

노트북은 어떤 runner를 어떤 인자로 호출할지 만 관리한다.


## Runner Config

manifest와 category를 먼저 정하고, runner와 summary viewer가 같은 설정을 공유하게 둔다.


In [ ]:
ACTIVE_BASELINES = ['PatchCore']  # add 'UniVAD' after its runtime deps/checkpoints are ready
CATEGORIES = ['breakfast_box']
AUTO_DISCOVER_MANIFESTS = True
MANIFEST_NAMES = [
    'query_motion_blur.jsonl',
    'query_low_light.jsonl',
    'query_gaussian_noise.jsonl',
]
EXCLUDED_MANIFEST_NAMES = {'query_identity.jsonl', 'query_multi.jsonl'}
WANDB_PROJECT = 'regram-condition-shift'
WANDB_MODE = 'disabled'  # switch to online only after WANDB_API_KEY is configured
SHOW_RUNNER_OUTPUT = True
RUNNER_OUTPUT_TAIL_LINES = 40

BASELINE_SPECS = {
    'PatchCore': {
        'runner_name': 'PatchCore manifest shift evaluation',
        'runner_path': EXP_ROOT / 'src' / 'core' / 'run_patchcore_manifest_shift.py',
        'runner_inputs': 'category + manifest jsonl(s) + raw LOCO dataset root',
        'runner_outputs': 'single summary json per category under reports/patchcore_manifest_shift',
        'report_subdir': 'patchcore_manifest_shift',
        'device': DEFAULT_PATCHCORE_DEVICE,
        'use_wandb': True,
        'wandb_group': 'patchcore',
        'wandb_log_images': True,
        'wandb_max_images': 2,
        'extra_args': [],
        'notes': 'PatchCore runner consumes raw LOCO directly and defaults to cpu when CUDA is unavailable.',
    },
    'UniVAD': {
        'runner_name': 'UniVAD manifest shift evaluation',
        'runner_path': EXP_ROOT / 'src' / 'univad' / 'run_manifest_shift.py',
        'runner_inputs': 'category + manifest jsonl(s) + raw LOCO dataset root',
        'runner_outputs': 'single summary json per category under reports/univad_manifest_shift',
        'report_subdir': 'univad_manifest_shift',
        'device': 'cuda',
        'use_wandb': True,
        'wandb_group': 'univad',
        'wandb_log_images': True,
        'wandb_max_images': 2,
        'extra_args': ['--image-size', '448', '--k-shot', '1', '--round', '0'],
        'notes': 'UniVAD requires CUDA plus prepared checkpoints/runtime deps under external/UniVAD.',
    },
}

if AUTO_DISCOVER_MANIFESTS:
    discovered = []
    for root in MANIFEST_ROOT_CANDIDATES:
        if root.exists():
            discovered.extend(path.name for path in sorted(root.glob('query_*.jsonl')))
    MANIFEST_NAMES = sorted({name for name in discovered if name not in EXCLUDED_MANIFEST_NAMES})

manifest_paths = resolve_manifest_paths(MANIFEST_NAMES)

run_configs = []
for baseline in ACTIVE_BASELINES:
    spec = BASELINE_SPECS[baseline]
    for category in CATEGORIES:
        summary_path = REPORT_ROOT / spec['report_subdir'] / f'{category}_multi_all.json'
        log_path = REPORT_ROOT / spec['report_subdir'] / 'logs' / f'{category}_multi_all.log.txt'
        runner_cmd = [
            sys.executable,
            str(spec['runner_path']),
            '--category', category,
            '--manifest', *[str(p) for p in manifest_paths],
            '--device', spec['device'],
            *spec['extra_args'],
        ]
        if spec['use_wandb']:
            runner_cmd.extend([
                '--use-wandb',
                '--wandb-project', WANDB_PROJECT,
                '--wandb-group', spec['wandb_group'],
                '--wandb-mode', WANDB_MODE,
            ])
            if spec['wandb_log_images']:
                runner_cmd.extend([
                    '--wandb-log-images',
                    '--wandb-max-images', str(spec['wandb_max_images']),
                ])
        run_configs.append({
            'baseline': baseline,
            'category': category,
            'device': spec['device'],
            'manifest_paths': list(manifest_paths),
            'manifest_names': list(MANIFEST_NAMES),
            'summary_path': summary_path,
            'log_path': log_path,
            'runner_cmd': runner_cmd,
            'use_wandb': spec['use_wandb'],
            'runner_name': spec['runner_name'],
            'runner_path': spec['runner_path'],
            'runner_inputs': spec['runner_inputs'],
            'runner_outputs': spec['runner_outputs'],
            'wandb_group': spec['wandb_group'],
            'wandb_log_images': spec['wandb_log_images'],
            'wandb_max_images': spec['wandb_max_images'],
            'notes': spec['notes'],
        })

display_title(
    'Runner Plan',
    f'Prepared `{len(run_configs)}` runs across `{len(ACTIVE_BASELINES)}` baselines and `{len(CATEGORIES)}` categories.',
)
display(pd.DataFrame([
    {
        'baseline': baseline,
        'runner_name': spec['runner_name'],
        'runner_path': str(spec['runner_path']),
        'runner_inputs': spec['runner_inputs'],
        'runner_outputs': spec['runner_outputs'],
        'device': spec['device'],
        'wandb_group': spec['wandb_group'] if spec['use_wandb'] else 'off',
        'wandb_mode': WANDB_MODE if spec['use_wandb'] else 'disabled',
        'wandb_log_images': spec['wandb_log_images'] if spec['use_wandb'] else False,
        'wandb_max_images': spec['wandb_max_images'] if spec['use_wandb'] else 0,
        'notes': spec['notes'],
    }
    for baseline, spec in BASELINE_SPECS.items() if baseline in ACTIVE_BASELINES
]))
display(pd.DataFrame({'manifest_name': MANIFEST_NAMES}))
display_run_plan(run_configs)


In [ ]:
display_title('Run Command Preview', 'Each run covers one baseline/category pair with all selected manifests in a single process.')
for config in run_configs:
    print('---')
    print('label =', format_run_label(config))
    print('manifest_count =', len(config['manifest_paths']))
    print('summary_path =', config['summary_path'])
    print('runner_cmd =', ' '.join(config['runner_cmd']))


In [ ]:
display_title(
    'External Baseline Setup',
    'PatchCore setup is lightweight. UniVAD assumes GPU runtime and prepared checkpoints under external/UniVAD.',
)

external_root = REPO_ROOT / 'external'
external_root.mkdir(parents=True, exist_ok=True)

if 'PatchCore' in ACTIVE_BASELINES:
    subprocess.run(['bash', '-lc', 'mkdir -p /content/ReGraM/external'], check=True)
    subprocess.run(
        [
            'bash', '-lc',
            'test -d /content/ReGraM/external/patchcore-inspection.clean/.git || '
            'git clone https://github.com/amazon-science/patchcore-inspection.git '
            '/content/ReGraM/external/patchcore-inspection.clean'
        ],
        check=True,
    )
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'faiss-cpu', 'timm'], check=True)

if 'UniVAD' in ACTIVE_BASELINES:
    subprocess.run(['bash', '-lc', 'mkdir -p /content/ReGraM/external'], check=True)
    subprocess.run(
        [
            'bash', '-lc',
            'test -d /content/ReGraM/external/UniVAD/.git || '
            'git clone https://github.com/FantasticGNU/UniVAD.git /content/ReGraM/external/UniVAD'
        ],
        check=True,
    )
    univad_root = REPO_ROOT / 'external' / 'UniVAD'
    checkpoint_root = univad_root / 'pretrained_ckpts'
    display(pd.DataFrame([
        {'key': 'univad_root_exists', 'value': univad_root.exists()},
        {'key': 'univad_checkpoint_root_exists', 'value': checkpoint_root.exists()},
        {'key': 'univad_ready', 'value': checkpoint_root.exists()},
        {'key': 'univad_note', 'value': 'If checkpoints or optional deps are missing, keep ACTIVE_BASELINES=['PatchCore'] until UniVAD runtime prep is complete.'},
    ]))


In [ ]:
run_history = []
total_runs = len(run_configs)
display_title('Execution Log', f'Started at `{now_string()}` with `{total_runs}` scheduled runs.')

for idx, config in enumerate(run_configs, start=1):
    label = format_run_label(config)
    started_at = now_string()
    print('=' * 100)
    print(f"[{idx}/{total_runs}] START {label} @ {started_at}")
    print('summary_path =', config['summary_path'])
    print('log_path =', config['log_path'])
    print('command =', ' '.join(config['runner_cmd']))
    tic = time.perf_counter()
    result = stream_subprocess(
        config['runner_cmd'],
        cwd=REPO_ROOT,
        show_output=SHOW_RUNNER_OUTPUT,
    )
    elapsed_sec = round(time.perf_counter() - tic, 2)
    status = 'success' if result['returncode'] == 0 else 'failed'
    finished_at = now_string()
    run_history.append({
        'order': idx,
        'baseline': config['baseline'],
        'category': config['category'],
        'status': status,
        'returncode': result['returncode'],
        'elapsed_sec': elapsed_sec,
        'started_at': started_at,
        'finished_at': finished_at,
        'summary_path': str(config['summary_path']),
        'log_path': str(config['log_path']),
    })
    print(f"[{idx}/{total_runs}] END   {label} -> {status} ({elapsed_sec}s) @ {finished_at}")
    if result['returncode'] != 0:
        print('-' * 100)
        print(f"Failure recap for {label}")
        print_output_tail('stderr', result['stderr'], tail_lines=RUNNER_OUTPUT_TAIL_LINES)
        if result['stdout'].strip():
            print_output_tail('stdout', result['stdout'], tail_lines=min(20, RUNNER_OUTPUT_TAIL_LINES))
        if Path(config['log_path']).exists():
            print(f"runner log file = {config['log_path']}")
        display(pd.DataFrame(run_history))
        raise RuntimeError(
            f"runner failed for {label}; inspect the streamed [stdout]/[stderr] lines above"
        )

print('=' * 100)
display_title('Execution Summary', f'Finished at `{now_string()}`.')
display(pd.DataFrame(run_history))


## Summary Viewer

runner가 남긴 `summary.json`과 `log.txt`를 기준으로만 결과를 본다.


In [ ]:
rows = []
clean_rows = []
for config in run_configs:
    summary_path = config['summary_path']
    if not summary_path.exists():
        display(Markdown(f"`summary.json` not found: `{summary_path}`"))
        continue

    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    payload = summary.get('payload', {})
    clean_rows.append({
        'baseline': summary['baseline'],
        'category': summary['class_name'],
        'clean_image_auroc': summary['metrics'].get('clean_image_auroc'),
        'clean_good_mean': summary['metrics'].get('clean_good_mean'),
        'clean_anomaly_mean': summary['metrics'].get('clean_anomaly_mean'),
    })
    severity_spec_by_cell = payload.get('severity_spec_by_cell', {})
    for aug_type, severity_map in payload.get('augmentations', {}).items():
        for severity, item in severity_map.items():
            cell_key = f'{aug_type}/{severity}'
            spec = severity_spec_by_cell.get(cell_key, payload.get('severity_spec', {}))
            rows.append({
                'baseline': summary['baseline'],
                'category': summary['class_name'],
                'shift_family': aug_type,
                'severity': severity,
                'mean': item.get('mean'),
                'fpr_over_clean_max': item.get('fpr_over_clean_max'),
                'mean_score_shift': item.get('mean_score_shift'),
                'image_auroc_vs_clean_anomaly': item.get('image_auroc_vs_clean_anomaly'),
                **{f'severity_param_{k}': v for k, v in spec.items()},
            })

df = pd.DataFrame(rows)
if not df.empty:
    df = df.sort_values(['baseline', 'category', 'shift_family', 'severity']).reset_index(drop=True)
clean_df = pd.DataFrame(clean_rows)
if not clean_df.empty:
    clean_df = clean_df.drop_duplicates(subset=['baseline', 'category']).sort_values(['baseline', 'category']).reset_index(drop=True)

display_title('Summary Viewer', 'Per-baseline, per-category summary with every (shift, severity) cell expanded.')
display(df if not df.empty else pd.DataFrame(columns=['baseline', 'category', 'shift_family', 'severity']))

display_title('Clean Reference Snapshot', 'Per-baseline clean metrics copied from each summary.')
display(clean_df if not clean_df.empty else pd.DataFrame(columns=['baseline', 'category']))

if run_history:
    display_title('Run History Snapshot')
    display(pd.DataFrame(run_history))


In [ ]:
plot_df = df.copy()
if plot_df.empty:
    display_title('Metric Tables', 'No summary rows found yet.')
else:
    plot_df['severity_rank'] = plot_df['severity'].map(SEVERITY_ORDER)
    plot_df = plot_df.sort_values(['baseline', 'shift_family', 'severity_rank']).reset_index(drop=True)

    metric_specs = [
        ('fpr_over_clean_max', 'Shifted FPR over clean max', 'FPR'),
        ('mean_score_shift', 'Shifted mean score shift', 'Score shift'),
        ('image_auroc_vs_clean_anomaly', 'Shifted vs anomaly AUROC', 'AUROC'),
    ]

    display_title('Metric Tables', 'Pivoted summary tables grouped by baseline (mean across categories if >1).')
    for baseline, baseline_df in plot_df.groupby('baseline'):
        display(Markdown(f'### {baseline}'))
        for metric_key, metric_title, _ in metric_specs:
            pivot = baseline_df.pivot_table(index='shift_family', columns='severity', values=metric_key, aggfunc='mean')
            ordered_cols = [col for col in ['low', 'medium', 'high'] if col in pivot.columns]
            pivot = pivot[ordered_cols]
            display(Markdown(f'#### {metric_title}'))
            display(pivot)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
        for ax, (metric_key, metric_title, y_label) in zip(axes, metric_specs):
            pivot = baseline_df.pivot_table(index='shift_family', columns='severity', values=metric_key, aggfunc='mean')
            ordered_cols = [col for col in ['low', 'medium', 'high'] if col in pivot.columns]
            pivot = pivot[ordered_cols]
            pivot.plot(kind='bar', ax=ax)
            ax.set_title(metric_title)
            ax.set_xlabel('shift_family')
            ax.set_ylabel(y_label)
            ax.legend(title='severity')
            ax.tick_params(axis='x', rotation=45)
        fig.suptitle(f'{baseline} summary', fontsize=14)
        plt.show()


In [ ]:
display_title('Recommended W&B Panels', 'Suggested panel recipes for the per-category multi-shift run schema.')
panel_specs = pd.DataFrame([
    {
        'panel_name': 'Worst FPR across all shift cells',
        'metric': 'worst_fpr_over_clean_max',
        'filter': 'project=regram-condition-shift',
        'group_by': 'group (= baseline), config.class_name',
        'notes': 'Sort Runs page by this scalar to see which (baseline, category) combo breaks hardest.',
    },
    {
        'panel_name': 'Mean FPR by severity',
        'metric': 'mean_fpr_by_severity/low, mean_fpr_by_severity/medium, mean_fpr_by_severity/high',
        'filter': 'project=regram-condition-shift',
        'group_by': 'group',
        'notes': 'Line chart of low/medium/high progression averaged over all 8 shift families.',
    },
    {
        'panel_name': 'Shift cells detail table',
        'metric': 'shift_cells (wandb.Table)',
        'filter': 'open individual run',
        'group_by': 'native table: shift, severity',
        'notes': 'Sort/filter 24 cells inside the run. Use wandb Compare mode to join tables across baselines.',
    },
    {
        'panel_name': 'Clean image AUROC across baselines',
        'metric': 'clean_image_auroc',
        'filter': 'project=regram-condition-shift',
        'group_by': 'group, config.class_name',
        'notes': 'Reference scalar — should be stable across runs of the same (baseline, category) pair.',
    },
    {
        'panel_name': 'Worst cell identity',
        'metric': 'worst_cell (string)',
        'filter': 'project=regram-condition-shift',
        'group_by': 'column in Runs table',
        'notes': 'Shows which shift/severity caused the worst FPR — quickest diagnostic column.',
    },
])
display(panel_specs)
